In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from my_krml_26044382.features.dates import add_date_features, add_lag_features, add_rolling_features
from my_krml_26044382.features.targets import create_future_targets

In [ ]:
df = pd.read_csv('../../data/interim/sydney_daily_2000_2025.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
df.shape

(9497, 23)

In [ ]:
df = df.drop(columns=['snowfall', 'rain'])

In [ ]:
df['temperature_range'] = df['temperature_2m_max'] - df['temperature_2m_min']
df['pressure_msl_change_1d'] = df['pressure_msl'].diff(1)
df['pressure_msl_change_3d'] = df['pressure_msl'].diff(3)

In [ ]:
df = add_date_features(df)

In [ ]:
lag_cols = [
    'cci', 'whi', 'temperature_2m', 'relative_humidity_2m', 'precipitation',
    'cloud_cover', 'wind_speed_10m', 'wind_gusts_10m', 'pressure_msl',
]
df = add_lag_features(df, lag_cols, lags=(1, 2, 3, 7))

In [ ]:
mean_cols = ['cci', 'whi', 'temperature_2m', 'relative_humidity_2m', 'cloud_cover', 'wind_gusts_10m']
df = add_rolling_features(df, mean_cols, windows=(3, 7, 14, 30), funcs=('mean',))

In [ ]:
df = add_rolling_features(df, ['precipitation'], windows=(3, 7, 14, 30), funcs=('sum',))

In [ ]:
df = create_future_targets(df, 'cci', [1, 2, 3])
df = create_future_targets(df, 'whc', 7)

In [ ]:
check = df.set_index('date')
d = pd.Timestamp('2010-01-01')
print(check.loc[d, 'cci_t_plus_1'], check.loc[d + pd.DateOffset(days=1), 'cci'])
print(check.loc[d, 'whc_t_plus_7'], check.loc[d + pd.DateOffset(days=7), 'whc'])

67.20104166666665 67.20104166666665
0.0 0


In [ ]:
target_cols = ['cci_t_plus_1', 'cci_t_plus_2', 'cci_t_plus_3', 'whc_t_plus_7']
feature_cols = [c for c in df.columns if c not in target_cols + ['date']]
print(f"{len(feature_cols)} features")

92 features


In [ ]:
df = df.dropna(subset=feature_cols).reset_index(drop=True)
print(df['date'].min(), df['date'].max(), df.shape)

2000-01-30 00:00:00 2025-12-31 00:00:00 (9468, 97)


In [ ]:
df[feature_cols].isna().sum().sum()

np.int64(0)

In [ ]:
df.to_csv('../../data/processed/features_daily.csv', index=False)

In [ ]:
import json

with open('../../models/feature_columns.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)